In [1]:
import pandas as pd
import sqlite3
import os


In [2]:
os.makedirs("raw", exist_ok=True)
os.makedirs("processed", exist_ok=True)
os.makedirs("output", exist_ok=True)


In [5]:
df = pd.read_csv("data.csv", encoding="latin1")


In [6]:
df.head()
df.info()
df.describe()
df.shape


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


(541909, 8)

In [7]:
df = df.drop_duplicates()


In [8]:
df.isnull().sum()
df = df.dropna()


In [9]:
df.columns = df.columns.str.lower().str.replace(" ", "_")


In [11]:
print(df.columns)


Index(['invoiceno', 'stockcode', 'description', 'quantity', 'invoicedate',
       'unitprice', 'customerid', 'country'],
      dtype='object')


In [12]:
df.head()


,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [13]:
df.columns = df.columns.str.strip()          # remove extra spaces
df.columns = df.columns.str.lower()         # lowercase
df.columns = df.columns.str.replace(" ", "_")


In [14]:
print(df.columns)


Index(['invoiceno', 'stockcode', 'description', 'quantity', 'invoicedate',
       'unitprice', 'customerid', 'country'],
      dtype='object')


In [16]:
df['invoicedate'] = pd.to_datetime(df['invoicedate'])
df['unitprice'] = df['unitprice'].astype(float)

In [18]:
# Standardize columns
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("Columns:", df.columns)

# Convert date column (adjust name if needed)
df['invoicedate'] = pd.to_datetime(df['invoicedate'])

# Convert price column
df['unitprice'] = df['unitprice'].astype(float)

Columns: Index(['invoiceno', 'stockcode', 'description', 'quantity', 'invoicedate',
       'unitprice', 'customerid', 'country'],
      dtype='object')


In [24]:
# Create Sales
df['sales'] = df['quantity'] * df['unitprice']

# Create Cost (assume 70% of selling price)
df['cost'] = df['sales'] * 0.7

# Create Margin
df['margin'] = df['sales'] - df['cost']

# Create High Value Customer Flag
df['high_value_customer'] = df['sales'].apply(lambda x: 1 if x > 1000 else 0)

In [25]:
df['margin'] = df['sales'] - df['cost']


In [26]:
df['high_value_customer'] = df['sales'].apply(lambda x: 1 if x > 1000 else 0)


In [27]:
df.to_csv("processed/processed_data.csv", index=False)


In [29]:
customers = df[['customerid']].drop_duplicates()
customers.to_csv("output/customers.csv", index=False)

In [31]:
orders = df[['invoiceno','invoicedate','customerid','sales']]
orders.to_csv("output/orders.csv", index=False)

In [33]:
products = df[['stockcode','description','unitprice']]
products.to_csv("output/products.csv", index=False)

In [34]:
conn = sqlite3.connect("output/database.sqlite")

customers.to_sql("customers", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)

conn.close()


In [38]:
# Load original raw file again for validation
raw_df = pd.read_csv("data.csv", encoding="latin1")

print("Original Shape:", raw_df.shape)
print("Processed Shape:", df.shape)

print("\nDuplicates Removed:", raw_df.shape[0] - df.shape[0])
print("\nMissing Values After Cleaning:\n", df.isnull().sum())

Original Shape: (541909, 8)
Processed Shape: (401604, 12)

Duplicates Removed: 140305

Missing Values After Cleaning:
 invoiceno              0
stockcode              0
description            0
quantity               0
invoicedate            0
unitprice              0
customerid             0
country                0
sales                  0
cost                   0
margin                 0
high_value_customer    0
dtype: int64


In [40]:
# Create Sales
df['sales'] = df['quantity'] * df['unitprice']

# Create Cost (Assume 70% of sales)
df['cost'] = df['sales'] * 0.7

# Create Margin
df['margin'] = df['sales'] - df['cost']

# High Value Customer Flag
df['high_value_customer'] = df['sales'].apply(lambda x: 1 if x > 1000 else 0)

In [41]:
df.to_csv("processed/processed_data.csv", index=False)


In [43]:
customers = df[['customerid', 'country']].drop_duplicates()
customers.to_csv("output/customers.csv", index=False)

In [44]:
import sqlite3

conn = sqlite3.connect("output/database.sqlite")

customers.to_sql("customers", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)

conn.close()


In [45]:
print("Customers Table Shape:", customers.shape)
print("Orders Table Shape:", orders.shape)
print("Products Table Shape:", products.shape)


Customers Table Shape: (4380, 2)
Orders Table Shape: (401604, 4)
Products Table Shape: (401604, 3)


In [46]:
from google.colab import files

files.download("processed/processed_data.csv")
files.download("output/database.sqlite")
files.download("output/customers.csv")
files.download("output/orders.csv")
files.download("output/products.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>